# 🔫 Notebook 3 — STONITH & Resource Fencing

> "Shoot The Other Node In The Head" — the most metal acronym in distributed systems.

## What you'll learn

- Why fencing tokens alone aren't always enough.
- **Resource fencing**: block the stale leader from the things it needs (network, disk, lock).
- **Node fencing (STONITH)**: forcibly power off / reboot the stale node.
- How these combine with token fencing in real systems (HDFS HA, Pacemaker, Patroni).


## 🛠️ Setup

```bash
cd 02-distributed-primitives/split-brain-and-fencing
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🤔 Why tokens aren't always enough

Fencing tokens work beautifully when **the shared resource can check the token**. But what if:

- The stale leader controls a **shared disk** and can write blocks the filesystem has no notion of tokens for.
- The stale leader sends **side-effectful commands** to outside systems (email, payments, IoT relays) that *don't* validate tokens.
- You're protecting something that's **not software** — a storage array exported via NFS, a SAN, or a physical robot.

For those cases, you need to physically prevent the stale leader from *doing* anything. That's resource fencing and STONITH.


## 🟦 Resource fencing: cut off access to the resource

Instead of trusting the stale leader to stop, we go to the **resource** (or the network between them) and revoke access.

Common techniques:

- Revoke the stale leader's NFS export (`exportfs -u`).
- Change a storage LUN's SCSI reservation so only the new leader can write.
- Close the stale leader's TCP connections at a firewall / load-balancer.
- Rotate a shared secret so stale leader's API calls start returning 401.


In [1]:
from dataclasses import dataclass, field
from typing import Dict, Set

@dataclass
class NFSExport:
    """Toy model of an NFS server that keeps an ACL of allowed clients."""
    name: str
    allowed: Set[str] = field(default_factory=set)
    files: Dict[str, str] = field(default_factory=dict)

    def mount(self, client: str) -> None:
        self.allowed.add(client)
        print(f"  mount  {self.name} -> allowed {sorted(self.allowed)}")

    def revoke(self, client: str) -> None:
        self.allowed.discard(client)
        print(f"  revoke {client} from {self.name} -> allowed {sorted(self.allowed)}")

    def write(self, client: str, path: str, content: str) -> bool:
        if client not in self.allowed:
            print(f"  ❌ {client} blocked by NFS ACL: cannot write {path}")
            return False
        self.files[path] = content
        print(f"  ✅ {client} wrote {path}: {content!r}")
        return True


nfs = NFSExport("/srv/shared")
nfs.mount("A")
nfs.write("A", "/srv/shared/state", "v1")

# Cluster promotes B. Before trusting B, we *fence* A at the resource level.
print("\n-- failover: revoke A, grant B --")
nfs.revoke("A")
nfs.mount("B")
nfs.write("B", "/srv/shared/state", "v2")

# A wakes up from a GC pause and tries to write.
print("\n-- stale A wakes up --")
nfs.write("A", "/srv/shared/state", "v1.1 (STALE)")


  mount  /srv/shared -> allowed ['A']
  ✅ A wrote /srv/shared/state: 'v1'

-- failover: revoke A, grant B --
  revoke A from /srv/shared -> allowed []
  mount  /srv/shared -> allowed ['B']
  ✅ B wrote /srv/shared/state: 'v2'

-- stale A wakes up --
  ❌ A blocked by NFS ACL: cannot write /srv/shared/state


False

Notice: the stale leader doesn't have to *cooperate* with being fenced. The resource (or the layer in front of it) simply refuses to serve it.

This is what HDFS calls **shared edits directory fencing** for the NameNode.


## 🟥 STONITH (node fencing): when in doubt, power it off

If you can't trust any per-resource check — for example the stale node has direct hardware access, multiple exports, or a "bad firmware day" — you just **take the whole node down**.

Common mechanisms:

- **IPMI / iDRAC / iLO** — out-of-band management cards; a single command power-cycles the node.
- **Managed PDUs** — cut power to the socket.
- **Hypervisor API** — `virsh destroy`, EC2 `StopInstances`, etc.
- **Watchdog timers** — a hardware watchdog reboots the node if the leader software stops petting it.

The new leader only starts serving writes **after** STONITH is confirmed successful. This removes all ambiguity: the old leader is *definitely* not writing, because it's off.


In [2]:
class IPMI:
    """Toy out-of-band power controller."""
    def __init__(self):
        self.power: Dict[str, bool] = {}

    def register(self, node: str) -> None:
        self.power[node] = True

    def is_up(self, node: str) -> bool:
        return self.power.get(node, False)

    def power_off(self, node: str) -> bool:
        print(f"  🔫 IPMI: powering off {node}")
        self.power[node] = False
        return True


class Node:
    def __init__(self, name: str, ipmi: IPMI):
        self.name = name
        self.ipmi = ipmi
        ipmi.register(name)

    def try_write(self, resource: list, value: str) -> None:
        if not self.ipmi.is_up(self.name):
            print(f"  💀 {self.name} is powered off; it can't do anything")
            return
        resource.append(f"{self.name}: {value}")
        print(f"  ✍️  {self.name} wrote {value!r}")


ipmi = IPMI()
A = Node("A", ipmi)
B = Node("B", ipmi)
shared_log: list = []

A.try_write(shared_log, "v1")

print("\n-- cluster loses contact with A; before electing B we STONITH A --")
ipmi.power_off("A")

B.try_write(shared_log, "v2")

print("\n-- even if A wakes up, it can't write: it is physically off --")
A.try_write(shared_log, "v1.1 (would have been STALE)")

print("\nshared log:")
for line in shared_log:
    print(" ", line)


  ✍️  A wrote 'v1'

-- cluster loses contact with A; before electing B we STONITH A --
  🔫 IPMI: powering off A
  ✍️  B wrote 'v2'

-- even if A wakes up, it can't write: it is physically off --
  💀 A is powered off; it can't do anything

shared log:
  A: v1
  B: v2


## 🧩 In practice: all three together

Real production systems layer these:

1. **Fencing tokens** (cheap, per-write) — the default everywhere we can teach the resource to check.
2. **Resource fencing** (medium cost) — for resources we can't modify (NFS, SAN, network).
3. **STONITH** (expensive, dramatic) — the last-resort hammer, used before promoting a new leader in HA clusters.

| System | Fencing tokens | Resource fencing | STONITH |
|---|---|---|---|
| HDFS NameNode HA | NameNode generation / epoch | revoke shared edits dir (NFS / QJM) | `sshfence` / IPMI if configured |
| Pacemaker / Corosync | (not primary mechanism) | SCSI3 PR reservations | ✅ IPMI, PDU, hypervisor |
| Patroni (Postgres HA) | xlog position / timeline | revoke replication slot | optional watchdog reboot |
| Kafka | controller epoch | — | — |
| ZooKeeper / etcd | zxid / mod_revision | — | — |

## 🎯 Rules of thumb

- **Can you change the resource?** Add a fencing token. Always. This is the simplest, safest layer.
- **Is the resource outside your control?** Add resource fencing: revoke credentials, rotate secrets, flip firewall rules.
- **Could a misbehaving node cause irreversible damage?** Add STONITH. Promote the new leader only after power-off is confirmed.

## ✅ You've now covered the whole topic

- 🧠 Notebook 1: reproduced the split-brain bug.
- 🛡️ Notebook 2: fixed it at the resource layer with **fencing tokens**.
- 🔫 Notebook 3: added the fallback layers — **resource fencing** and **STONITH**.

Further reading:

- Martin Kleppmann, *How to do distributed locking* — https://martin.kleppmann.com/2016/02/08/how-to-do-distributed-locking.html
- *Designing Data-Intensive Applications*, chapter 8 ("The Truth Is Defined by the Majority").
- ZooKeeper `zxid`, Kafka controller epoch, HDFS NameNode HA, Pacemaker STONITH docs.
